# RAG systems

Author: Umberto Michelucci, umberto.michelucci@hslu.ch

## Minimal Retrieval-Augmented Generation (RAG)

This notebook shows the simplest possible implementation of a RAG system.

A RAG system works in 3 steps:

1. Store documents
2. Retrieve the most relevant documents for a question
3. Use an LLM to answer using the retrieved context

The goal is to help understand the core idea behind modern AI assistants that use external knowledge.

![Embeddings Diagram](RAG1.png)x

## Import libraries and create the OpenAI client

We import:
- `OpenAI` for API calls
- `numpy` for similarity calculations

In [11]:
from openai import OpenAI
from pathlib import Path
import numpy as np
from pypdf import PdfReader

# Path to the file on the Desktop
key_path = Path.home() / "Desktop" / "api-key.txt"

# Read the key
api_key = key_path.read_text().strip()

# Create client
client = OpenAI(api_key=api_key)

In [12]:
def read_pdf(path):
    reader = PdfReader(path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

## Create a tiny knowledge base

This is our "database".

Each document contains one piece of bioengineering knowledge.

In a real RAG system:
- these could come from PDFs
- scientific papers
- textbooks
- websites
- databases

In [2]:
# 1. Tiny knowledge base
documents = [
    "Tissue engineering combines cells, scaffolds, and biochemical signals to repair or replace damaged tissue.",
    "Hydrogels are water-rich polymer networks often used as scaffolds because they can mimic soft biological tissues.",
    "CRISPR-Cas9 is a genome editing technology that can cut DNA at targeted locations.",
    "Biosensors combine a biological recognition element with a physical transducer to detect molecules.",
    "Drug delivery systems aim to release therapeutic molecules at the right place, time, and dose."
]

## Create embeddings

Embeddings convert text into numerical vectors.

Texts with similar meaning produce similar vectors.

The embedding model does NOT answer questions.
It only converts text into a mathematical representation.

In [3]:
# 2. Function to create embeddings
def embed(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding)

## Embed all documents

We now compute one embedding vector for each document.

This step is usually done once and stored in a vector database.

In [6]:
# 3. Embed all documents
doc_embeddings = [embed(doc) for doc in documents]

## Retrieve the most relevant documents

We now implement semantic search.

Steps:
1. Embed the user question
2. Compare the question vector to all document vectors
3. Compute cosine similarity
4. Return the most similar documents

Cosine similarity measures how close two vectors are.

In [7]:
# 4. Retrieve the most relevant documents
def retrieve(query, k=2):
    query_embedding = embed(query)

    similarities = []
    for doc_embedding in doc_embeddings:
        score = np.dot(query_embedding, doc_embedding) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
        )
        similarities.append(score)

    top_indices = np.argsort(similarities)[-k:][::-1]

    return [documents[i] for i in top_indices]

In [8]:
# 5. Generate answer using retrieved context
def rag_answer(question):
    context_docs = retrieve(question, k=2)

    context = "\n".join(context_docs)

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    return response.output_text

## Test retrieval

We first test ONLY the retrieval system.

This allows us to see:
- which documents were selected
- where the information comes from
- similarity scores

This is extremely important in RAG systems.

In [9]:
# 6. Try it
question = "Why are hydrogels useful in tissue engineering?"

answer = rag_answer(question)

print(answer)

Hydrogels are useful in tissue engineering because they are water-rich polymer networks that can mimic soft biological tissues, making them effective as scaffolds for repairing or replacing damaged tissue.


## What happened internally?

The pipeline is:

Question
↓
Embedding
↓
Similarity search
↓
Retrieved documents
↓
LLM prompt with context
↓
Generated answer

This architecture is the foundation of many modern AI assistants.

# RAG from PDFs

In [14]:
pdf_1_text = read_pdf("paper1.pdf")
pdf_2_text = read_pdf("paper2.pdf")

In [16]:
documents = [
    pdf_1_text,
    pdf_2_text
]

# For Google Colab

    from google.colab import files

    uploaded = files.upload()

## Careful with large documents!

In [17]:
# 3. Embed all documents
doc_embeddings = [embed(doc) for doc in documents]

BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'input': maximum context length is 8192 tokens.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [18]:
def read_pdf(path):

    reader = PdfReader(path)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

![Embeddings Diagram](RAG2.png)

In [19]:
def chunk_text(text, chunk_size=500):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):

        chunk = " ".join(words[i:i + chunk_size])

        chunks.append(chunk)

    return chunks

In [21]:
documents = []
document_names = []

chunks_1 = chunk_text(pdf_1_text)
chunks_2 = chunk_text(pdf_2_text)

for chunk in chunks_1:

    documents.append(chunk)
    document_names.append("paper1.pdf")

for chunk in chunks_2:

    documents.append(chunk)
    document_names.append("paper2.pdf")

Why chunking matters

Without chunking:

1 PDF = 1 huge embedding

With chunking:

1 PDF → many small chunks → many embeddings

This is the core idea of practical RAG systems.

In [22]:
print("Number of chunks:", len(documents))

Number of chunks: 29


In [23]:
# 3. Embed all documents
doc_embeddings = [embed(doc) for doc in documents]

In [24]:
# 6. Try it
question = "What is the leading cause of cardiovascular disease?"

answer = rag_answer(question)

print(answer)

The leading cause of cardiovascular disease (CVD) in Asian countries is hypertension (high blood pressure). The population-attributable fraction of hypertension for CVD is as high as 60% in Asian countries. High blood pressure, often accompanied by high salt intake, is a major contributor to the prominence of stroke in Asia compared to coronary heart disease (CHD).


# RAG System Version 2.0 - Where and in which document is the information

In [26]:
from pypdf import PdfReader

def build_documents_from_pdf(path, chunk_size=500):
    reader = PdfReader(path)

    docs = []
    names = []
    pages = []
    chunks = []

    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text()

        if page_text:

            words = page_text.split()

            for chunk_number, i in enumerate(range(0, len(words), chunk_size), start=1):

                chunk = " ".join(words[i:i + chunk_size])

                docs.append(chunk)
                names.append(path)
                pages.append(page_number)
                chunks.append(chunk_number)

    return docs, names, pages, chunks

In [27]:
docs1, names1, pages1, chunks1 = build_documents_from_pdf("paper1.pdf")
docs2, names2, pages2, chunks2 = build_documents_from_pdf("paper2.pdf")

documents = docs1 + docs2
document_names = names1 + names2
document_pages = pages1 + pages2
document_chunks = chunks1 + chunks2

In [28]:
def retrieve(query, k=2):

    query_embedding = embed(query)

    similarities = []

    for doc_embedding in doc_embeddings:

        score = np.dot(query_embedding, doc_embedding) / (
            np.linalg.norm(query_embedding) *
            np.linalg.norm(doc_embedding)
        )

        similarities.append(score)

    top_indices = np.argsort(similarities)[-k:][::-1]

    results = []

    for i in top_indices:

        results.append({
            "document_id": i,
            "document_name": document_names[i],
            "page": document_pages[i],
            "chunk": document_chunks[i],
            "document": documents[i],
            "similarity": similarities[i]
        })

    return results

In [29]:
def rag_answer(question):

    retrieved_docs = retrieve(question, k=2)

    context = ""

    for r in retrieved_docs:
        context += f"""
Source: {r["document_name"]}, page {r["page"]}, chunk {r["chunk"]}
Text: {r["document"]}
"""

    prompt = f"""
Answer the question using ONLY the context below.

After the answer, cite the source using:
(document name, page number, chunk number)

Context:
{context}

Question:
{question}
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    print("=== RETRIEVED SOURCES ===\n")

    for r in retrieved_docs:
        print("DOCUMENT:", r["document_name"])
        print("PAGE:", r["page"])
        print("CHUNK:", r["chunk"])
        print("SIMILARITY:", round(r["similarity"], 3))
        print("TEXT PREVIEW:", r["document"][:500], "...")
        print()

    print("=== GENERATED ANSWER ===\n")
    print(response.output_text)

In [30]:
question = "What is the leading cause of cardiovascular disease?"

rag_answer(question)

=== RETRIEVED SOURCES ===

DOCUMENT: paper1.pdf
PAGE: 5
CHUNK: 1
SIMILARITY: 0.449
TEXT PREVIEW: High Smoking Rate The smoking rate for men in Asian countries (except Singa- pore, Hong Kong, and India) in 2000 remains high at 40% to 60%, 1,46 – 49 although it has declined substantially over the last 2 decades (Figure 4).8,50 –54 The smoking rate for women in all Asian countries is far lower, at 3% to 15%, than in Western countries. 1,46 – 49 It is a specific characteristic in Asia that women smoke and drink much less than men. 1,46 – 49 Because smoking is a potent risk factor not only for C ...

DOCUMENT: paper1.pdf
PAGE: 4
CHUNK: 2
SIMILARITY: 0.447
TEXT PREVIEW: in ( ) 1989 to 1993. The incidence rates of 6 Japanese populations and a single Chinese population were far lower than those of selected Western countries. The world standard population was used for the calculation of age-adjusted rates for 35- to 64-year-olds. Reprinted with permission from the Japan Atherosclerosis Society.